# CNN Audio Classifier for MagnaTagATune
## Deep Learning Model for Music Tagging

This notebook implements a CNN-based audio classifier using mel spectrograms.

In [ ]:
!pip install -q torch torchaudio librosa numpy pandas matplotlib scikit-learn tqdm tensorboard

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchaudio
import torchaudio.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Model Architecture

VGG-style CNN for audio classification with mel spectrogram input.

In [ ]:
class AudioCNN(nn.Module):
    """VGG-style CNN for audio tagging."""
    
    def __init__(self, n_classes=50, n_mels=128, dropout=0.5):
        super().__init__()
        
        self.n_mels = n_mels
        
        # Mel spectrogram transform
        self.mel_transform = T.MelSpectrogram(
            sample_rate=22050,
            n_fft=2048,
            hop_length=512,
            n_mels=n_mels
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        # CNN layers (VGG-style)
        self.conv_layers = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )
    
    def forward(self, x):
        # x: (batch, samples)
        
        # Extract mel spectrogram
        x = self.mel_transform(x)  # (batch, n_mels, time)
        x = self.amplitude_to_db(x)
        x = x.unsqueeze(1)  # (batch, 1, n_mels, time)
        
        # Normalize
        x = (x - x.mean()) / (x.std() + 1e-8)
        
        # CNN
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        
        return x

# Test model
model = AudioCNN(n_classes=50).to(device)
test_input = torch.randn(2, 22050 * 29).to(device)
output = model(test_input)
print(f"Model output shape: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Data Augmentation

In [ ]:
class AudioAugmentation:
    """Audio augmentation transforms."""
    
    def __init__(self, sr=22050, p=0.5):
        self.sr = sr
        self.p = p
    
    def time_stretch(self, waveform, rate_range=(0.8, 1.2)):
        """Time stretch augmentation."""
        if np.random.random() < self.p:
            rate = np.random.uniform(*rate_range)
            # Using torchaudio's time stretch
            stretch = T.TimeStretch(hop_length=512, n_freq=1025)
            spec = torch.stft(waveform, n_fft=2048, hop_length=512, return_complex=True)
            stretched = stretch(spec, rate)
            waveform = torch.istft(stretched, n_fft=2048, hop_length=512)
        return waveform
    
    def pitch_shift(self, waveform, semitones_range=(-2, 2)):
        """Pitch shift augmentation (simplified version)."""
        if np.random.random() < self.p:
            # For actual pitch shifting, use librosa or torchaudio.functional
            pass
        return waveform
    
    def add_noise(self, waveform, noise_level=0.005):
        """Add random noise."""
        if np.random.random() < self.p:
            noise = torch.randn_like(waveform) * noise_level
            waveform = waveform + noise
        return waveform
    
    def random_gain(self, waveform, gain_range=(0.8, 1.2)):
        """Random gain adjustment."""
        if np.random.random() < self.p:
            gain = np.random.uniform(*gain_range)
            waveform = waveform * gain
        return waveform
    
    def __call__(self, waveform):
        waveform = self.add_noise(waveform)
        waveform = self.random_gain(waveform)
        return waveform

augment = AudioAugmentation()
print("Augmentation pipeline ready!")

## 3. Training Pipeline

In [ ]:
class Trainer:
    """Training class for audio models."""
    
    def __init__(self, model, train_loader, val_loader, 
                 lr=1e-4, weight_decay=1e-5, device='cuda'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.AdamW(
            model.parameters(), 
            lr=lr, 
            weight_decay=weight_decay
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=100
        )
        
        self.history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        
        pbar = tqdm(self.train_loader, desc='Training')
        for batch_idx, (data, target) in enumerate(pbar):
            data, target = data.to(self.device), target.to(self.device)
            
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        return total_loss / len(self.train_loader)
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_targets = []
        
        for data, target in tqdm(self.val_loader, desc='Validating'):
            data, target = data.to(self.device), target.to(self.device)
            
            output = self.model(data)
            loss = self.criterion(output, target)
            total_loss += loss.item()
            
            preds = torch.sigmoid(output)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(target.cpu().numpy())
        
        all_preds = np.vstack(all_preds)
        all_targets = np.vstack(all_targets)
        
        # Calculate AUC for each class
        aucs = []
        for i in range(all_targets.shape[1]):
            if all_targets[:, i].sum() > 0:  # Only if positive samples exist
                auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
                aucs.append(auc)
        
        mean_auc = np.mean(aucs) if aucs else 0
        
        return total_loss / len(self.val_loader), mean_auc
    
    def train(self, epochs=100, save_best=True):
        best_auc = 0
        
        for epoch in range(epochs):
            print(f"\nEpoch {epoch + 1}/{epochs}")
            
            train_loss = self.train_epoch()
            val_loss, val_auc = self.validate()
            
            self.scheduler.step()
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_auc'].append(val_auc)
            
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")
            
            if save_best and val_auc > best_auc:
                best_auc = val_auc
                torch.save(self.model.state_dict(), 'best_model.pt')
                print(f"Best model saved! AUC: {best_auc:.4f}")
        
        return self.history

print("Trainer class ready!")

## 4. Synthetic Dataset for Testing

In [ ]:
class SyntheticAudioDataset(Dataset):
    """Synthetic dataset for testing the pipeline."""
    
    def __init__(self, n_samples=1000, n_classes=50, sr=22050, duration=5.0):
        self.n_samples = n_samples
        self.n_classes = n_classes
        self.sr = sr
        self.duration = duration
        self.audio_length = int(sr * duration)
        
        # Generate random labels
        self.labels = (np.random.rand(n_samples, n_classes) > 0.9).astype(np.float32)
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        # Generate synthetic audio (sine waves with noise)
        t = np.linspace(0, self.duration, self.audio_length)
        
        # Random frequencies based on labels
        active_labels = np.where(self.labels[idx] > 0)[0]
        frequencies = 100 + active_labels * 50  # Different freq per label
        
        audio = np.zeros(self.audio_length)
        for freq in frequencies[:3]:  # Use up to 3 frequencies
            audio += np.sin(2 * np.pi * freq * t) * 0.3
        
        # Add noise
        audio += np.random.randn(self.audio_length) * 0.1
        
        audio = torch.tensor(audio, dtype=torch.float32)
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return audio, labels

# Create synthetic datasets
train_dataset = SyntheticAudioDataset(n_samples=800, duration=5.0)
val_dataset = SyntheticAudioDataset(n_samples=200, duration=5.0)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

## 5. Train Model (Demo)

In [ ]:
# Initialize model and trainer
model = AudioCNN(n_classes=50, n_mels=128).to(device)
trainer = Trainer(model, train_loader, val_loader, lr=1e-4, device=device)

# Train for a few epochs (demo)
history = trainer.train(epochs=3, save_best=True)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()

axes[1].plot(history['val_auc'], label='Validation AUC', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation AUC')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Model Export

In [ ]:
# Export model to ONNX for web deployment
def export_to_onnx(model, output_path='audio_classifier.onnx'):
    model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 22050 * 5).to(device)
    
    # Export
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        input_names=['audio'],
        output_names=['tags'],
        dynamic_axes={
            'audio': {0: 'batch_size'},
            'tags': {0: 'batch_size'}
        },
        opset_version=14
    )
    print(f"Model exported to {output_path}")

# Uncomment to export
# export_to_onnx(model)